# 1. Integration of 1 million (RGS) * 36 times (RDS) random sampling results to obtain the importance score of genes in a single gene set within a single dataset
> Input: Clustering performance evaluation metrics for each RDS from 1 million iterations

> Intermediate processing steps:

> (1) For each indicator in each RDS, sort from largest to smallest, select the top % RGS, and consider them effective in the primary and metastatic classification. Mark the corresponding matrix values as 1, and set the other RGS as 0, generating a 0-1 matrix.

> (2) Organize the same indicator from each RDS into a matrix; assess the similarity of matrices from different indicators in the same dataset. If the matrices are similar, merge them (matrix addition—if any matrix indicator has a value of 1, the corresponding position in the merged matrix will also be 1); if the matrices are not similar, keep them separately.

> (3) For each dataset, select the RGS that can effectively differentiate primary and metastatic samples in 90% of the RDS from the 0-1 matrix for each indicator. Then, perform a frequency count of these selected RGS across all indicators in the dataset. This frequency represents the gene importance score. Sort the genes by their importance scores from highest to lowest.

> Output: Gene ranking for each dataset's 25 gene sets, along with gene importance rankings.

In [ ]:
Top_10_RGS <- function(df) { 
  n_rows <- nrow(df)

  result_matrix <- apply(df, 2, function(col) {  
    top_n <- ceiling(0.1 * n_rows)
    idx <- order(col, decreasing = TRUE)[1:top_n]
    out <- integer(n_rows)
    out[idx] <- 1
    out
  })
  result_matrix <- as.matrix(result_matrix)
  rownames(result_matrix) <- rownames(df)
  colnames(result_matrix) <- colnames(df)
  return(result_matrix)
}

In [ ]:
geneList = c('Cluster1','Cluster2','Cluster3','Cluster4','Cluster5','Cluster6','Cluster7','Cluster8','Cluster9','Cluster10',
             'Cluster11','Cluster12','Cluster13','Cluster14','Cluster15','Cluster16','Cluster17','Cluster18','Cluster19','Cluster20','H1','H2','H3','H4','H5')

dataList = c('GSE125989','GSE60542','GSE62321','GSE73168','GSE85258','GSE44408') 

ALLGeneOrder = list()

for (g in geneList){
    cat("==== Processing gene set:", g, "====\n")  
    ALLGeneOrder[[g]] = list()  
    
    for (d in dataList){
        
        Benchmark <- paste0("./Project/100wSampling/",g,"/",d,"/Benchmark.csv")
        GenesSampledRes <- paste0("./Project/100wSampling/",g,"/",d,"/GenesSampledRes.csv")
        
        if (!file.exists(Benchmark) | !file.exists(GenesSampledRes)) {
            message("Skip (file missing): ", g, " / ", d)
            next
        }
        RDS_4index <- tryCatch(read.csv(Benchmark, row.names = 1),error = function(e) NULL)  # Read performance metrics                
        RGSlist <- tryCatch(read.csv(GenesSampledRes, row.names = 1),error = function(e) NULL) # Read the randomly sampled gene set RDS list
        if (is.null(RDS_4index) | is.null(RGSlist) |
            nrow(RDS_4index) == 0 | nrow(RGSlist) == 0) {
            message("Skip (empty or read error): ", g, " / ", d)
            next
        }

        # (1) For each dataset and gene set, build 4 unit matrices for the 4 performance metrics
        ACC <- RDS_4index[, grep("acc", colnames(RDS_4index), value = TRUE)]
        AUC <- RDS_4index[, grep("auc", colnames(RDS_4index), value = TRUE)]
        MCC <- RDS_4index[, grep("mcc", colnames(RDS_4index), value = TRUE)]
        Kappa <- RDS_4index[, grep("kappa", colnames(RDS_4index), value = TRUE)]
        # Select the top 10% of RGS in each metric as the effective RGS and build unit matrices
        ACC_matrix <- Top_10_RGS(ACC)
        ACC_matrix <- Top_10_RGS(ACC)
        AUC_matrix <- Top_10_RGS(AUC)
        MCC_matrix <- Top_10_RGS(MCC)
        Kappa_matrix <- Top_10_RGS(Kappa)

        # (2) Integrate the results of multiple metrics to construct the gene set's importance ranking in a bulk dataset
        # Combine the unit matrices from the 4 metrics, where an RGS is considered effective if any metric indicates it has distinguishing ability
        all_index = ACC_matrix + AUC_matrix + MCC_matrix + Kappa_matrix
        all_index = ACC_matrix+AUC_matrix+MCC_matrix+Kappa_matrix
        all_index[all_index > 1] <- 1 
        # Select RGS that are effective in more than 90% of RDS from all metrics
        selected_RGS <- rownames(all_index)[rowMeans(all_index) > 0.9]
        # Retrieve the specific genes for the selected RGS and merge them into a single gene set
        GeneSet = c(unlist(RGSlist[rownames(RGSlist) %in% selected_RGS,], use.names = FALSE))

        if (length(GeneSet) == 0) {
          ALLGeneOrder[[g]][[d]] <- numeric(0)
          next
        }
        # Count the frequency of each gene's occurrence as its importance score, and then sort the gene list by the score
        geneOrder <- names(sort(table(GeneSet), decreasing = TRUE)) 

        ALLGeneOrder[[g]][[d]] = geneOrder
    }
}

# 2. Integration of Gene Ranks from Multiple Datasets to Obtain a Comprehensive Gene Importance Ranking

> Input: 25 gene sets with importance rankings from multiple datasets

> Output: A comprehensive gene set with 25 genes ranked by importance

In [ ]:
source("./Function/2.2.RankAggregate.RRA.R")

In [ ]:
setList = list()
for(g in names(ALLGeneOrder)){
    setList[[g]] = RankAggregate.RRA (ALLGeneOrder[[g]],unlist(geneSet[[g]]))
}

# 3. Identify the Optimal Gene Set Subset for Classification in Lymph Node Metastasis Data from Breast Cancer

> Input: ① A comprehensive gene set of 25 genes ranked by importance; ② Lymph node metastasis data from breast cancer

> Intermediate processing steps:

> (1) Forward feature selection method: Sequentially use the top n genes from each gene set as classifier features and predict sample categories using SVM and LR classifiers. Calculate classification performance metrics for different feature sets.

> (2) Based on classification performance metrics, select the top % genes from each gene set as the final feature gene set.

> Output: A multi-feature gene set that can be used to identify metastasis potential subpopulations in single-cell data

In [ ]:
source("./Function/2.3.4.EvaluateSignature.R")

In [ ]:
setSelect = list() 
for (g in names(setList) ){
    cat("==== Processing gene set:", g, "====\n")
    set = setList[[g]]
    setSelect[[g]]<-Classifier(expr=GSE44408_exp, pData=pData44408,genelist=set$Name)
}